# AMML Ranking Competition 2026 — solution

**Preferred:** Kaggle Notebook **GPU**.  
**Metric:** NDCG@10  

Pipeline (course-aligned):
1. Lexical/structural features (BM25, TF-IDF, overlap, …)
2. **GroupKFold OOF** fine-tune of `DiTy/cross-encoder-russian-msmarco` via HF `AutoModelForSequenceClassification` (keeps MS MARCO head; no ST.fit)
3. CatBoost **YetiRank** on lexical φ + CE features (no post-hoc leaky blend)

Fine-tuning uses **only** competition train labels.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
print('platform:', platform.platform())
print('python:', sys.version)
print('on_kaggle:', ON_KAGGLE)

if ON_KAGGLE:
    OUT = Path('/kaggle/working')
    MODELS = OUT / 'models'
    candidates = [
        Path('/kaggle/input/amml-ranking-competition-2026'),
        Path('/kaggle/input/competitions/amml-ranking-competition-2026'),
    ]
    DATA = next((p for p in candidates if (p / 'train.csv').exists()), None)
    if DATA is None:
        hits = list(Path('/kaggle/input').rglob('train.csv'))
        DATA = hits[0].parent if hits else candidates[0]
    CODE_DS = Path('/kaggle/input/amml-ranking-2026-code')
    if (CODE_DS / 'src').exists():
        ROOT = CODE_DS
    else:
        ROOT = OUT / 'amml-ranking-competition-2026'
        if not ROOT.exists():
            subprocess.check_call([
                'git', 'clone', '--depth', '1',
                'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git',
                str(ROOT),
            ])
    try:
        import rank_bm25  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25==0.2.2'])
    try:
        import sentence_transformers  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==3.3.1'])
    import catboost, torch
    print('catboost', catboost.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
else:
    ROOT = Path('.').resolve()
    DATA = ROOT / 'data'
    OUT = ROOT / 'results'
    MODELS = ROOT / 'models'

sys.path.insert(0, str(ROOT))
# Prefer newest code: dataset if it has HF DiTy helper, else shallow clone to /tmp.
if ON_KAGGLE:
    code_ok = False
    if (CODE_DS / 'src' / 'ce_train.py').exists():
        sys.path.insert(0, str(CODE_DS))
        try:
            from src.ce_train import oof_hf_sequence_ce_scores  # noqa: F401
            ROOT = CODE_DS
            code_ok = True
            print('using CODE_DS with oof_hf_sequence_ce_scores')
        except Exception as e:
            print('CODE_DS stale:', e)
            sys.path = [p for p in sys.path if p != str(CODE_DS)]
    if not code_ok:
        repo = Path('/tmp/amml-ranking-competition-2026')
        if repo.exists():
            import shutil; shutil.rmtree(repo, ignore_errors=True)
        subprocess.check_call([
            'git', 'clone', '--depth', '1',
            'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git',
            str(repo),
        ])
        sys.path.insert(0, str(repo))
        ROOT = repo
        print('using git clone', repo)
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print('ROOT', ROOT)
print('DATA', DATA)
assert (DATA / 'train.csv').exists(), list(Path('/kaggle/input').glob('*')) if ON_KAGGLE else DATA

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.preprocessing import prepare_frame
from src.features import assemble_features
from src.train import cross_validate, train_full, predict, get_feature_cols
from src.ce_train import oof_hf_sequence_ce_scores
from src.metrics import mean_ndcg_at_k

train = prepare_frame(pd.read_csv(DATA / 'train.csv'))
test = prepare_frame(pd.read_csv(DATA / 'test.csv'))
print(train.shape, test.shape)

print('Building lexical/structural features...')
feat_tr, feat_te = assemble_features(train, test)

# DiTy Russian MS MARCO CE head via transformers (reliable on Kaggle GPU).
CE_MODEL = 'DiTy/cross-encoder-russian-msmarco'
ce_epochs = 2 if torch.cuda.is_available() else 0
ce_bs = 16 if torch.cuda.is_available() else 8
print(f'HF-CE OOF model={CE_MODEL} epochs={ce_epochs} bs={ce_bs}')
ce_oof, ce_te = oof_hf_sequence_ce_scores(
    train, test,
    models_dir=MODELS,
    model_name=CE_MODEL,
    epochs=ce_epochs,
    batch_size=ce_bs,
    max_len=320,
    lr=1e-5,
    n_splits=5,
)

# CE features + within-query rank/z + raw CE as ranking candidate
feat_tr = feat_tr.copy()
feat_te = feat_te.copy()
feat_tr['ce_score'] = ce_oof
feat_te['ce_score'] = ce_te
for df in (feat_tr, feat_te):
    g = df.groupby('query_id')['ce_score']
    df['ce_score_qrank'] = g.rank(method='average', ascending=False)
    mean = g.transform('mean')
    std = g.transform('std').replace(0, np.nan)
    df['ce_score_qz'] = ((df['ce_score'] - mean) / std).fillna(0.0)

# Native CatBoost text features (query/title/passage)
for feat, raw in ((feat_tr, train), (feat_te, test)):
    feat['query_text'] = raw['query'].astype(str).to_numpy()
    feat['title_text'] = raw['title'].fillna('').astype(str).to_numpy()
    feat['passage_text'] = raw['text'].fillna('').astype(str).to_numpy()

labels = train['relevance']
from catboost import CatBoostRanker, Pool
from src.features import CAT_FEATURES
from src.train import FEATURE_EXCLUDE, default_params, make_pool

TEXT_COLS = ['query_text', 'title_text', 'passage_text']

def get_cols(df):
    return [c for c in df.columns if c not in FEATURE_EXCLUDE]

def make_rich_pool(df, cols, with_label=True):
    g = df['query_id'].astype('category').cat.codes.to_numpy()
    order = np.argsort(g, kind='mergesort')
    X = df.iloc[order][cols]
    cat_idx = [cols.index(c) for c in CAT_FEATURES if c in cols]
    text_idx = [cols.index(c) for c in TEXT_COLS if c in cols]
    kw = dict(cat_features=cat_idx, text_features=text_idx, group_id=g[order])
    if with_label:
        return Pool(X, df.iloc[order]['relevance'].to_numpy(), **kw)
    return Pool(X, **kw), order

print('CatBoost YetiRank: lexical + text + DiTy CE...')
from sklearn.model_selection import GroupKFold
params = default_params()
params.update(dict(iterations=2500, learning_rate=0.03, depth=8, border_count=128))
df = feat_tr.copy(); df['relevance'] = labels.to_numpy()
cols = get_cols(df)
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(df))
fold_scores = []
for fold, (tr_i, va_i) in enumerate(gkf.split(df, df['relevance'], groups=df['query_id'])):
    tr = df.iloc[tr_i].reset_index(drop=True)
    va = df.iloc[va_i].reset_index(drop=True)
    tr_pool = make_rich_pool(tr, cols, True)
    vg = va['query_id'].astype('category').cat.codes.to_numpy()
    vo = np.argsort(vg, kind='mergesort')
    vs = va.iloc[vo]
    cat_idx = [cols.index(c) for c in CAT_FEATURES if c in cols]
    text_idx = [cols.index(c) for c in TEXT_COLS if c in cols]
    va_pool = Pool(vs[cols], vs['relevance'].to_numpy(), group_id=vg[vo], cat_features=cat_idx, text_features=text_idx)
    model = CatBoostRanker(**params)
    model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
    pred = np.empty(len(va)); pred[vo] = model.predict(va_pool)
    oof[va_i] = pred
    sc = va.copy(); sc['prediction'] = pred
    nd = mean_ndcg_at_k(sc); fold_scores.append(nd)
    print(f'fold {fold}: NDCG@10={nd:.6f} best_iter={model.get_best_iteration()}')
cv = float(np.mean(fold_scores))
print('CV', cv, 'std', float(np.std(fold_scores)))

# CE-only OOF for comparison / possible blend of OOF ranks (no leak)
ce_scored = train[['id','query_id','relevance']].copy(); ce_scored['prediction'] = ce_oof
print('CE-only OOF NDCG@10', mean_ndcg_at_k(ce_scored))

full_params = {**params, 'iterations': 2000}
full_params.pop('od_type', None); full_params.pop('od_wait', None)
df_full = feat_tr.copy(); df_full['relevance'] = labels.to_numpy()
model = CatBoostRanker(**full_params)
model.fit(make_rich_pool(df_full, cols, True))
te_pool, te_order = make_rich_pool(feat_te, cols, False)
pred = np.empty(len(feat_te)); pred[te_order] = model.predict(te_pool)

# Also dump CE-only submission (strong baseline candidate)
sub_ce = test[['id']].copy(); sub_ce['prediction'] = ce_te
sub_ce.to_csv(OUT / 'submission_ce_only.csv', index=False)

sub = test[['id']].copy(); sub['prediction'] = pred
sub.to_csv(OUT / 'submission.csv', index=False)
print('wrote', OUT / 'submission.csv', 'and', OUT / 'submission_ce_only.csv')
print('OOF stack NDCG@10', mean_ndcg_at_k(train[['id','query_id','relevance']].assign(prediction=oof)), 'CV', cv)
sub.head()